[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hanenalmayouf/applied-ml-workshop/blob/main/labs_colab/day1/lab_1_manafeth_leakage_solution.ipynb)

<div dir="rtl">

# 🧭 مختبر اليوم الأول — كشف التسريب وصياغة مشكلة مغادرة العملاء

**ورشة أسس تعلم الآلة التطبيقي — اليوم 1 من 5**

هذا الدفتر مبني على مواصفات مختبرات «منافذ» المعتمدة للدورة، ومُجهَّز للعمل مباشرة في **Google Colab** أو في Jupyter محليًا.

**كيف تفتحه في Colab:** اضغط زر «Open in Colab» أعلى هذا الدفتر (أو أعلى الـ README المرافق له). ستُحمَّل بيانات هذا اليوم تلقائيًا من مستودع الدورة على GitHub بمجرد تشغيل خلية إعداد البيانات — بلا أي رفع يدوي. فقط إذا تعذّر الاتصال بالإنترنت داخل Colab لأي سبب، ستظهر خانة احتياطية لرفع ملف `manafeth_data_package.zip` يدويًا.

> 📌 راجع `00_start_here.md` قبل البدء لمعرفة طريقة استخدام خلايا **فكّر أولًا** و**TODO** و**مساعدة** في هذه الدفاتر.

</div>

In [ ]:
from pathlib import Path
import pandas as pd

# 1) نبحث عن مجلد البيانات بجانب هذا الدفتر (يعمل محليًا، أو في Colab إذا رفعت المجلد كاملًا)
DATA_CANDIDATES = [Path("manafeth_data_package"), Path("data"), Path("../data/raw"), Path("data/raw")]
DATA_DIR = next((p for p in DATA_CANDIDATES if p.exists()), None)

# 2) إذا لم نجد المجلد ونحن داخل Google Colab (فتحت الدفتر بزر Open in Colab ولم يُرفَق مجلد البيانات
#    تلقائيًا)، نحمّل حزمة بيانات هذا اليوم مباشرة من نفس المستودع على GitHub — بلا أي تدخل منك
if DATA_DIR is None:
    try:
        import google.colab  # يفشل الاستيراد إن لم نكن داخل Colab، وننتقل إلى except أدناه
        import urllib.request
        import zipfile

        zip_url = "https://raw.githubusercontent.com/hanenalmayouf/applied-ml-workshop/main/labs_colab/day1/manafeth_data_package.zip"
        zip_name = "manafeth_data_package.zip"
        print("لم يتم العثور على مجلد البيانات محليًا — يجري تحميلها تلقائيًا من مستودع الدورة على GitHub...")
        try:
            urllib.request.urlretrieve(zip_url, zip_name)
            with zipfile.ZipFile(zip_name) as z:
                z.extractall(".")
            DATA_DIR = next((p for p in DATA_CANDIDATES if p.exists()), None)
            if DATA_DIR is not None:
                print("تم تحميل البيانات تلقائيًا بنجاح ✅")
        except Exception as download_error:
            print("تعذّر التحميل التلقائي من GitHub:", download_error)

        # خطة بديلة فقط إذا فشل التحميل التلقائي (مثلًا بلا اتصال إنترنت داخل Colab)
        if DATA_DIR is None:
            from google.colab import files

            print("سنطلب منك رفع حزمة البيانات يدويًا كخطة بديلة — ارفع ملف manafeth_data_package.zip المرفق مع هذا المختبر.")
            uploaded = files.upload()
            for name in uploaded:
                if name.lower().endswith(".zip"):
                    with zipfile.ZipFile(name) as z:
                        z.extractall(".")
            DATA_DIR = next((p for p in DATA_CANDIDATES if p.exists()), Path("manafeth_data_package"))
    except ImportError:
        DATA_DIR = Path("manafeth_data_package")
        print("تنبيه: لسنا داخل Google Colab ولم يوجد مجلد بيانات — ضع حزمة البيانات بجانب الدفتر.")

CUSTOMERS_PATH = DATA_DIR / "manafeth_customers.parquet"
ORDERS_PATH = DATA_DIR / "manafeth_orders.parquet"
VEHICLES_PATH = DATA_DIR / "markabat_listings_sample.csv"
SHIFTED_PATH = DATA_DIR / "shifted_month.parquet"

print("مجلد البيانات المستخدم:", DATA_DIR.resolve())
assert CUSTOMERS_PATH.exists(), "تعذّر العثور على manafeth_customers.parquet — تأكد من رفع حزمة البيانات كاملة."

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl

mpl.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.figsize"] = (7, 4)

<div dir="rtl">

## 🎯 هدف المختبر

تستكشف جدول العملاء الفعلي وتحوّل طلب العمل إلى مسألة تصنيف محددة: **هل سيتوقف العميل عن إجراء طلبات مكتملة خلال الثلاثين يومًا التالية لتاريخ اللقطة؟** كما تتعلم قاعدة الوقت التي تمنع إدخال معلومات من المستقبل إلى النموذج.

## السيناريو والبيانات

يحتوي `manafeth_customers.parquet` على **48,000 صف**؛ كل صف يمثل **عميلًا واحدًا عند تاريخ لقطة شهري**. الهدف هو `churned_30d`. القيمة `1` تعني أن العميل لم يجرِ طلبًا مكتملًا في الثلاثين يومًا التالية، والقيمة `0` تعني أنه استمر في الطلب.

| نوع العمود | أعمدة واقعية من الملف | القرار |
|---|---|---|
| هدف | `churned_30d` | يبقى في `y` فقط |
| معرّف | `customer_id` | يُستبعد من الخصائص |
| خصائص آمنة مبدئيًا | `city`, `city_tier`, `device`, `payment_method`, `tenure_months`, `orders_per_month`, `avg_basket_sar`, `days_since_last_order`, `distinct_categories`, `promo_usage_rate`, `avg_rating`, `last_promo_used` | مرشحة لتدخل `X` بعد الفحص |
| **تسريب معلومات** | `refund_issued`, `support_ticket_after_snapshot`, `next_month_orders` | تُستبعد قطعًا — تحدث بعد اللقطة |
| تواريخ | `signup_date`, `snapshot_date` | لا تُستخدم كخصائص في هذا المختبر |

</div>

<div dir="rtl">

## 🤔 فكّر أولًا

قبل تشغيل أي كود: اقرأ وصف عمود `next_month_orders`. هل تتوفر قيمته لحظة اتخاذ قرار المتابعة مع العميل؟ لماذا لا يصلح كخاصية رغم أنه قد يرفع دقة النموذج ظاهريًا؟

</div>

In [ ]:
customers = pd.read_parquet(CUSTOMERS_PATH)
customers.head()

In [ ]:
customers.info()
customers.isna().sum().sort_values(ascending=False)

In [ ]:
customers["churned_30d"].value_counts(normalize=True)

<div dir="rtl">

### 📊 رسم: توزيع الهدف

مثال مُنفَّذ — لاحظ أن فئة `churned_30d = 1` أقل تكرارًا (بيانات غير متوازنة)، وهذا سيهمّنا في اليوم الرابع.

</div>

In [ ]:
counts = customers["churned_30d"].value_counts().sort_index()
plt.bar(["استمر (0)", "غادر (1)"], counts.values, color=["#33CC99", "#F26522"])
plt.title("توزيع هدف مغادرة العملاء خلال 30 يومًا")
plt.ylabel("عدد العملاء")
plt.show()

<div dir="rtl">

### 💡 مساعدة

إذا أردت رؤية أي الأعمدة فيها قيم ناقصة بصريًا لا رقميًا فقط، ارسم عدد القيم الناقصة لكل عمود كمخطط أعمدة أفقي.

</div>

In [ ]:
missing = customers.isna().sum()
missing = missing[missing > 0].sort_values()
missing.plot(kind="barh", color="#5B4FCF")
plt.title("عدد القيم الناقصة لكل عمود")
plt.xlabel("عدد الصفوف الناقصة")
plt.show()

<div dir="rtl">

## 📝 التصنيف: صنّف كل عمود

اختبر كل عمود مشكوك فيه بالسؤال: **«هل كانت هذه القيمة موجودة عند تاريخ اللقطة قبل بدء الثلاثين يومًا التالية؟»**. ثم أكمل القائمتين التاليتين.

</div>

In [ ]:
safe_features = [
    "city", "city_tier", "device", "payment_method",
    "tenure_months", "orders_per_month", "avg_basket_sar",
    "days_since_last_order", "distinct_categories", "promo_usage_rate",
    "avg_rating", "last_promo_used"
]

leak_columns = [
    "refund_issued",
    "support_ticket_after_snapshot",
    "next_month_orders"
]

print("خصائص آمنة:", safe_features)
print("أعمدة مسرّبة مستبعدة:", leak_columns)

<div dir="rtl">

## 🧠 صياغة المشكلة

اكتب في الخلية التالية (كنص Markdown) جملة بصيغة: «الصف يمثل …، والهدف هو …، والقرار الذي يساعده النموذج هو …».

</div>

<div dir="rtl">

_اكتب صياغتك هنا._

</div>

<div dir="rtl">

## النتيجة المتوقعة

يجب أن تلاحظ أن `avg_rating` و`last_promo_used` يحتويان على قيم ناقصة، وأن فئة المغادرة أقل من فئة الاستمرار. الأهم أن تسجّل قرارًا واضحًا باستبعاد الأعمدة الثلاثة المسرّبة، لا أن تكتفي بالقول إنها «أعمدة غير مناسبة».

## المهارات التي راجعتها

صياغة مشكلة عمل، تحديد وحدة التحليل والهدف والخصائص، قراءة جدول Parquet، فحص أنواع الأعمدة والنقص، وتطبيق اختبار الوقت لاكتشاف تسريب البيانات.

</div>

<div dir="rtl">

## ✅ تحقق ذاتيًا قبل إغلاق الدفتر
- [ ] `safe_features` لا يحتوي على `customer_id` ولا على أي عمود تاريخ
- [ ] `leak_columns` يحتوي على الأعمدة الثلاثة بالضبط: `refund_issued`, `support_ticket_after_snapshot`, `next_month_orders`
- [ ] كتبت صياغة المشكلة بجملة واحدة واضحة
- [ ] لاحظت عدم توازن فئة الهدف من الرسم البياني

**غدًا:** نبني خط معالجة (Pipeline) يعالج القيم الناقصة والأعمدة العددية والفئوية باستخدام `safe_features` نفسها.

</div>